# M12 — Combined Fine-Grained Guidance with Global–Local MaxViT Fusion

**EEEM068 Applied Machine Learning — MaxViT-Tiny Combined Fine-Grained Guidance Experiment**

**Scientific question:** Does combining detection-derived candidate localisation,
pseudo-segmentation guidance and global-local feature fusion improve diabetic-retinopathy
grading compared with using each component independently?

This is a **combined-component experiment**, not another isolated component test. M12 uses
**M07 as the training reference** (the same controls: architecture, seed, epochs, LR, loss,
selection rule, preprocessing) and **M11 as the main code template** (audit-before-cache
pattern, shared-backbone multi-view fusion, gradient accumulation, corrected AMP loop).

**Architecture — three views through one shared MaxViT-Tiny backbone:**

```
Global RGB image (M07/M11's global view) ──────────┐
Local candidate crop (M11's crop-selection method) ─┼─ shared MaxViT backbone
Pseudo-mask-guided RGB image (M10's mask applied    │
to the global image, still 3 channels)  ────────────┘
        |
  concatenate 3 embeddings
        |
     dropout
        |
  linear classifier -> 5 DR grades
```

A **single shared backbone** processes all three views (not three independent backbones) —
this keeps the parameter increase modest and keeps the comparison against M07/M09/M10/M11
controlled rather than confounded by a large, unrelated capacity increase.

The candidate detector and pseudo-segmentation masks used throughout this notebook are
**heuristic** and are **not validated against expert lesion annotations**. The generated
regions are described as candidate regions / pseudo-masks / heuristic fine-grained guidance /
lesion-like structures — never as verified lesions, ground-truth lesion segmentation, or
clinically validated lesion crops.

**Internal test set loaded: False** — the test split is never loaded anywhere in this
notebook. M12 is still a validation-stage experiment; the test set remains reserved for the
final frozen comparison after a possible M13.

| Setting | M12 value |
|---|---|
| Model | MaxViT-Tiny (shared backbone, three views) |
| Pretraining | ImageNet |
| Image size | 224 × 224 |
| Seed | 42 |
| Maximum epochs | 25 |
| Warm-up epochs | 3 |
| Early-stopping patience | 7 |
| Learning rate | 5e-5 |
| Weight decay | 0.05 |
| Gradient clipping | 1.0 |
| Optimizer | AdamW |
| Scheduler | Linear warm-up + cosine decay |
| Loss | Clipped class-weighted CE + 0.5 × ordinal penalty |
| Class-weight range | [0.5, 2.5] |
| Selection rule | Validation QWK first, macro-F1 tie-break, then lower validation loss |
| Preprocessing | P0 |
| Physical batch size | 8 |
| Gradient accumulation | 4 |
| Effective batch size | 32 |
| Internal test set | Never loaded |

## 2. Imports and reproducibility

In [ ]:
import json
import random
from pathlib import Path
from itertools import combinations

import cv2
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

from torchvision import transforms
import torchvision.transforms.functional as TF

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.utils.class_weight import compute_class_weight

print("Imports OK.")

In [ ]:
SEED = 42


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed()
print(f"Seed set: {SEED}")

## 3. Paths and configuration

`M07_LOG_DIR` is M12's training reference. `M09_LOG_DIR`/`M10_LOG_DIR`/`M11_LOG_DIR` are used
for the required comparisons (Section 24) and failure-overlap analysis (Section 25) — all
read-only, no checkpoints from any of them are ever loaded.

In [ ]:
EXPERIMENT_ID = "M12"
RUN_NAME = "M12_combined_fine_grained_guidance"
EXPERIMENT_TYPE = "combined_fine_grained_guidance_ablation"
REFERENCE_EXPERIMENT = "M07"
TEMPLATE_EXPERIMENT = "M11"
PREDECESSOR_EXPERIMENT = "M07_ordinal_aware_loss"

MODEL_NAME = "maxvit_tiny_tf_224.in1k"
IMAGE_SIZE = 224
NUM_CLASSES = 5
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

VIT_MEAN = (0.485, 0.456, 0.406)
VIT_STD = (0.229, 0.224, 0.225)

# Three views need more GPU memory than M11's two, hence a smaller physical batch. Effective
# batch size is kept identical to M07/M11 (32) via a larger accumulation factor.
PHYSICAL_BATCH_SIZE = 8
GRAD_ACCUMULATION_STEPS = 4
EFFECTIVE_BATCH_SIZE = PHYSICAL_BATCH_SIZE * GRAD_ACCUMULATION_STEPS

LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.05

MAX_EPOCHS = 25
WARMUP_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 7

LABEL_SMOOTHING = 0.0
GRAD_CLIP = 1.0
QWK_TOLERANCE = 1e-4
NUM_WORKERS = 4

TRAINING_MODE = "single_stage"
CLASSIFIER_DROPOUT = 0.3

USE_CLASS_WEIGHTS = True
USE_WRS = False
CLASS_WEIGHT_METHOD = "balanced"
CLASS_WEIGHT_CLIP_MIN = 0.5
CLASS_WEIGHT_CLIP_MAX = 2.5

ORDINAL_PENALTY_WEIGHT = 0.5

# Exact M07 safe-retinal augmentation.
HORIZONTAL_FLIP_PROBABILITY = 0.5
ROTATION_DEGREES = 7
BRIGHTNESS_JITTER = 0.05
CONTRAST_JITTER = 0.05
SATURATION_JITTER = 0.02
HUE_JITTER = 0.0
TRANSLATE_FRACTION = 0.02
SCALE_MIN = 0.98
SCALE_MAX = 1.02

# Same candidate/mask thresholds as M09/M10/M11 - a scientific control, not a dependency on
# any of their trained results.
DETECTION_SIZE = 512
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)
BACKGROUND_SIGMA = 12.0
DARK_RESPONSE_PERCENTILE = 97.0

MIN_COMPONENT_AREA = 3
MAX_COMPONENT_AREA = 350
MAX_ASPECT_RATIO = 4.0
MAX_CANDIDATE_COMPONENTS = 12

LOCAL_CROP_SIZE = 256  # M11's local-crop rule

PROJECT_ROOT = Path(
    "/scratch/New AML/EEEM068-LSA-Diabetic-Retinopathy"
)

SPLITS_DIR = PROJECT_ROOT / "configs" / "splits"
TRAIN_CSV = SPLITS_DIR / "train_split.csv"
VAL_CSV = SPLITS_DIR / "val_split.csv"

LOG_DIR = PROJECT_ROOT / "logs" / "maxvit_tiny" / RUN_NAME
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "maxvit_tiny" / RUN_NAME
FIGURE_DIR = PROJECT_ROOT / "results" / "figures" / "maxvit_tiny" / RUN_NAME
for directory in [LOG_DIR, CHECKPOINT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / "best.pt"

M07_LOG_DIR = PROJECT_ROOT / "logs" / "maxvit_tiny" / PREDECESSOR_EXPERIMENT
M09_LOG_DIR = PROJECT_ROOT / "logs" / "maxvit_tiny" / "M09_detection_guided_regions"
M10_LOG_DIR = PROJECT_ROOT / "logs" / "maxvit_tiny" / "M10_pseudo_segmentation_guidance"
M11_LOG_DIR = PROJECT_ROOT / "logs" / "maxvit_tiny" / "M11_global_local_crop_fusion"

# Reuse M11's crop-metadata cache and M10's mask cache where the configuration matches
# exactly (validated in Sections 7/8). M12 also has its own cache for anything specific to
# the combined guided view.
M11_CROP_PARAM_KEY = (
    f"det{DETECTION_SIZE}_clahe{CLAHE_CLIP_LIMIT}_bg{BACKGROUND_SIGMA}_"
    f"pct{DARK_RESPONSE_PERCENTILE}_area{MIN_COMPONENT_AREA}-{MAX_COMPONENT_AREA}_"
    f"ar{MAX_ASPECT_RATIO}_comp{MAX_CANDIDATE_COMPONENTS}_crop{LOCAL_CROP_SIZE}"
)
M11_CROP_CACHE_DIR = PROJECT_ROOT / "cache" / "maxvit_tiny" / "M11_local_crops" / M11_CROP_PARAM_KEY

M10_MASK_PARAM_KEY = (
    f"det{DETECTION_SIZE}_clahe{CLAHE_CLIP_LIMIT}_bg{BACKGROUND_SIGMA}_"
    f"pct{DARK_RESPONSE_PERCENTILE}_area{MIN_COMPONENT_AREA}-{MAX_COMPONENT_AREA}_"
    f"ar{MAX_ASPECT_RATIO}_comp{MAX_CANDIDATE_COMPONENTS}"
)
M10_MASK_CACHE_DIR = PROJECT_ROOT / "cache" / "maxvit_tiny" / "M10_pseudo_masks" / M10_MASK_PARAM_KEY

M12_CACHE_DIR = PROJECT_ROOT / "cache" / "maxvit_tiny" / "M12_guided_views"
M12_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment : {EXPERIMENT_ID} / {RUN_NAME}")
print(f"Reference  : {REFERENCE_EXPERIMENT}  |  Code template: {TEMPLATE_EXPERIMENT}")
print(f"PHYSICAL_BATCH={PHYSICAL_BATCH_SIZE}  ACCUM={GRAD_ACCUMULATION_STEPS}  "
      f"EFFECTIVE_BATCH={EFFECTIVE_BATCH_SIZE}")
print(f"Logs -> {LOG_DIR}")
print(f"M11 crop cache -> {M11_CROP_CACHE_DIR}")
print(f"M10 mask cache -> {M10_MASK_CACHE_DIR}")
print(f"M12 own cache  -> {M12_CACHE_DIR}  (git-ignored; not committed)")

## 4. Device and AMP settings

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

if device.type != "cuda":
    raise RuntimeError(
        "CUDA GPU is not available. Do not start this training run on CPU."
    )

use_amp = device.type == "cuda"

# Use a controlled initial AMP scale so overflow-related scale reductions
# are easier to track consistently from the beginning of training.
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp,
    init_scale=1024.0,
)

print(
    f"AMP enabled: {use_amp}  "
    f"initial scale: {scaler.get_scale() if use_amp else 'n/a'}"
)

## 5. Load train/validation manifests

Only `train` and `validation` splits are ever loaded — an explicit guard below asserts this.

In [ ]:
loaded_split_names = {"train": TRAIN_CSV, "validation": VAL_CSV}

for split_name in loaded_split_names:
    assert split_name in {"train", "validation"}
assert "test" not in loaded_split_names

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

required_columns = {"image", "level", "patient_id", "eye", "filepath"}
for split_name, frame in [("train", train_df), ("validation", val_df)]:
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {sorted(missing_columns)}")
    missing_files = frame.loc[~frame["filepath"].map(lambda path: Path(path).exists())]
    if len(missing_files) > 0:
        raise FileNotFoundError(f"{split_name} contains {len(missing_files):,} missing image files.")

print(f"Train      : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print("Internal test set loaded: False")

## 6. Leakage and split checks

In [ ]:
train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])
assert train_patients.isdisjoint(val_patients), "Patient overlap detected between train and validation."

train_images = set(train_df["image"])
val_images = set(val_df["image"])
assert train_images.isdisjoint(val_images), "Image overlap detected between train and validation."

print(f"Train patients      : {len(train_patients):,}")
print(f"Validation patients : {len(val_patients):,}")
print("Patient overlap check : PASSED (zero overlap)")
print("Image overlap check   : PASSED (zero overlap)")

## P0 preprocessing, candidate detection, local-crop selection, and pseudo-mask generation

Shared building blocks — identical logic to M09 (mask generation), M10 (exact-pixel soft
mask), and M11 (component detection, fixed local-crop selection). Reproduced here (not
imported, since each notebook is self-contained) so that M12 can validate the existing M10/M11
caches against this exact logic, and regenerate anything missing using the same code.

In [ ]:
def _load_rgb(path: str) -> np.ndarray:
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        raise ValueError(f"Cannot decode image: {path}")
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def _crop_black_borders(img: np.ndarray, threshold: int = 10) -> np.ndarray:
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    margin = int(min(w, h) * 0.02)
    x, y = max(0, x - margin), max(0, y - margin)
    w = min(img.shape[1] - x, w + 2 * margin)
    h = min(img.shape[0] - y, h + 2 * margin)
    return img[y:y + h, x:x + w]


def _pad_square(img: np.ndarray) -> np.ndarray:
    h, w = img.shape[:2]
    side = max(h, w)
    canvas = np.zeros((side, side, 3), dtype=img.dtype)
    y = (side - h) // 2
    x = (side - w) // 2
    canvas[y:y + h, x:x + w] = img
    return canvas


def preprocess_p0_full(image_path: str) -> np.ndarray:
    img = _load_rgb(image_path)
    img = _crop_black_borders(img)
    img = _pad_square(img)
    return img


def _detect_candidate_components(image_rgb_full: np.ndarray) -> dict:
    """Identical thresholds/logic to M09/M10/M11."""
    detection_image = cv2.resize(
        image_rgb_full, (DETECTION_SIZE, DETECTION_SIZE), interpolation=cv2.INTER_AREA
    )
    green = detection_image[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID_SIZE)
    enhanced_green = clahe.apply(green)
    background = cv2.GaussianBlur(enhanced_green, (0, 0), BACKGROUND_SIGMA)
    dark_response = cv2.subtract(background, enhanced_green)

    gray = cv2.cvtColor(detection_image, cv2.COLOR_RGB2GRAY)
    field_mask = (gray > 10).astype(np.uint8)
    dark_response = dark_response * field_mask

    field_pixels = dark_response[field_mask > 0]
    if field_pixels.size == 0 or field_pixels.max() <= 0:
        threshold_value = 255.0
        binary = np.zeros_like(field_mask, dtype=np.uint8)
    else:
        threshold_value = float(np.percentile(field_pixels, DARK_RESPONSE_PERCENTILE))
        binary = (
            (dark_response >= threshold_value) & (dark_response > 0) & (field_mask > 0)
        ).astype(np.uint8) * 255

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)

    candidates = []
    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]
        w = stats[label_id, cv2.CC_STAT_WIDTH]
        h = stats[label_id, cv2.CC_STAT_HEIGHT]
        if area < MIN_COMPONENT_AREA or area > MAX_COMPONENT_AREA:
            continue
        aspect_ratio = max(w, h) / max(1, min(w, h))
        if aspect_ratio > MAX_ASPECT_RATIO:
            continue
        response_strength = float(dark_response[labels == label_id].mean())
        cx, cy = centroids[label_id]
        candidates.append({
            "label_id": int(label_id), "area": int(area), "score": response_strength,
            "centroid_x": float(cx), "centroid_y": float(cy),
        })

    candidates.sort(key=lambda c: c["score"], reverse=True)
    accepted_components = candidates[:MAX_CANDIDATE_COMPONENTS]

    accepted_component_mask_512 = np.zeros_like(binary, dtype=np.uint8)
    for component in accepted_components:
        accepted_component_mask_512[labels == component["label_id"]] = 255

    return {
        "dark_response": dark_response, "threshold_value": threshold_value, "labels": labels,
        "accepted_components": accepted_components,
        "accepted_component_mask_512": accepted_component_mask_512,
        "detection_image": detection_image,
    }


def select_local_crop(image_rgb_full: np.ndarray) -> dict:
    """Identical to M11's fixed 256x256 centroid-centred crop selection. Never receives
    the label."""
    detection = _detect_candidate_components(image_rgb_full)
    accepted_components = detection["accepted_components"]
    detection_image = detection["detection_image"]

    fallback_used = len(accepted_components) == 0
    if fallback_used:
        centre_x = DETECTION_SIZE / 2.0
        centre_y = DETECTION_SIZE / 2.0
        selected_rank, selected_area, selected_score = -1, 0, 0.0
    else:
        selected = accepted_components[0]
        centre_x, centre_y = selected["centroid_x"], selected["centroid_y"]
        selected_rank, selected_area, selected_score = 0, selected["area"], selected["score"]

    half = LOCAL_CROP_SIZE // 2
    x1 = max(0, min(int(round(centre_x - half)), DETECTION_SIZE - LOCAL_CROP_SIZE))
    y1 = max(0, min(int(round(centre_y - half)), DETECTION_SIZE - LOCAL_CROP_SIZE))
    x2, y2 = x1 + LOCAL_CROP_SIZE, y1 + LOCAL_CROP_SIZE

    return {
        "crop_512": detection_image[y1:y2, x1:x2],
        "num_candidate_components": len(accepted_components),
        "selected_component_rank": selected_rank, "selected_component_area": selected_area,
        "selected_component_score": selected_score,
        "selected_component_centroid_x": centre_x, "selected_component_centroid_y": centre_y,
        "crop_x1": x1, "crop_y1": y1, "crop_x2": x2, "crop_y2": y2,
        "fallback_used": fallback_used, "detection_image": detection_image,
        "accepted_component_mask_512": detection["accepted_component_mask_512"],
    }


def generate_pseudo_segmentation_mask(image_rgb_full: np.ndarray) -> dict:
    """Identical to M10: exact accepted-component pixels, area-interpolated soft resize,
    never re-thresholded."""
    detection = _detect_candidate_components(image_rgb_full)
    accepted_components = detection["accepted_components"]
    labels = detection["labels"]

    pseudo_mask_512 = np.zeros_like(detection["accepted_component_mask_512"], dtype=np.uint8)
    accepted_areas = []
    for component in accepted_components:
        pseudo_mask_512[labels == component["label_id"]] = 255
        accepted_areas.append(component["area"])

    pseudo_mask_224 = cv2.resize(
        pseudo_mask_512.astype(np.float32) / 255.0,
        (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA,
    )
    pseudo_mask_224 = np.clip(pseudo_mask_224, 0.0, 1.0).astype(np.float32)

    return {
        "mask": pseudo_mask_224,
        "num_components": len(accepted_components),
        "mask_coverage": float(pseudo_mask_224.mean()),
        "threshold_value": float(detection["threshold_value"]),
        "accepted_component_areas": accepted_areas,
    }


print("Shared P0/detection/crop/mask functions ready (identical logic to M09/M10/M11).")

## 7. Load or validate local-crop metadata

Reuses M11's crop-metadata cache if the crop-selection configuration matches exactly
(checked via the parameter-derived cache-directory name itself). Validates: every
train/validation image has an entry, cache paths exist, and image stems are unique (a
duplicate stem would silently overwrite another image's cached crop).

In [ ]:
def _crop_cache_path_for(image_name: str) -> Path:
    return M11_CROP_CACHE_DIR / f"{Path(image_name).stem}.json"


all_image_names = pd.concat([train_df["image"], val_df["image"]], ignore_index=True)
all_cache_stems = all_image_names.map(lambda name: Path(str(name)).stem)
if not all_cache_stems.is_unique:
    duplicate_stems = all_cache_stems[all_cache_stems.duplicated(keep=False)].unique()
    raise RuntimeError(
        f"Duplicate image stems would collide in the crop cache: {duplicate_stems[:10].tolist()}"
    )
print("Crop-cache image-stem uniqueness check: PASSED")

crop_cache_valid = M11_CROP_CACHE_DIR.exists()
if crop_cache_valid:
    missing_crop_entries = [
        row["image"] for _, row in pd.concat([train_df, val_df]).iterrows()
        if not _crop_cache_path_for(row["image"]).exists()
    ]
    if missing_crop_entries:
        print(f"M11 crop cache is missing {len(missing_crop_entries):,} entries; will fill them on demand.")
    else:
        print(f"M11 crop cache found and complete for all {len(train_df) + len(val_df):,} images -> {M11_CROP_CACHE_DIR}")
else:
    print(f"M11 crop cache not found at {M11_CROP_CACHE_DIR}; will build entries on demand.")


def get_crop_metadata(row) -> dict:
    cache_path = _crop_cache_path_for(row["image"])
    if cache_path.exists():
        with open(cache_path) as f:
            return json.load(f)
    image_full = preprocess_p0_full(row["filepath"])
    crop_data = select_local_crop(image_full)
    metadata = {
        key: crop_data[key] for key in [
            "num_candidate_components", "selected_component_rank", "selected_component_area",
            "selected_component_score", "selected_component_centroid_x",
            "selected_component_centroid_y", "crop_x1", "crop_y1", "crop_x2", "crop_y2",
            "fallback_used",
        ]
    }
    M11_CROP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "w") as f:
        json.dump(metadata, f)
    return metadata

## 8. Load or validate pseudo-mask metadata

Reuses M10's pseudo-mask cache under the same validation discipline as Section 7.

In [ ]:
def _mask_cache_path_for(image_name: str) -> Path:
    return M10_MASK_CACHE_DIR / f"{Path(image_name).stem}.npz"


mask_cache_valid = M10_MASK_CACHE_DIR.exists()
if mask_cache_valid:
    missing_mask_entries = [
        row["image"] for _, row in pd.concat([train_df, val_df]).iterrows()
        if not _mask_cache_path_for(row["image"]).exists()
    ]
    if missing_mask_entries:
        print(f"M10 mask cache is missing {len(missing_mask_entries):,} entries; will fill them on demand.")
    else:
        print(f"M10 mask cache found and complete for all {len(train_df) + len(val_df):,} images -> {M10_MASK_CACHE_DIR}")
else:
    print(f"M10 mask cache not found at {M10_MASK_CACHE_DIR}; will build entries on demand.")


def get_pseudo_mask(row) -> np.ndarray:
    cache_path = _mask_cache_path_for(row["image"])
    if cache_path.exists():
        with np.load(cache_path) as cached:
            return cached["mask"].astype(np.float32)
    image_full = preprocess_p0_full(row["filepath"])
    mask_data = generate_pseudo_segmentation_mask(image_full)
    M10_MASK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_path,
        mask=mask_data["mask"].astype(np.float32),
        num_components=np.int16(mask_data["num_components"]),
        mask_coverage=np.float32(mask_data["mask_coverage"]),
        threshold_value=np.float32(mask_data["threshold_value"]),
    )
    return mask_data["mask"].astype(np.float32)


def get_mask_metadata(row) -> dict:
    cache_path = _mask_cache_path_for(row["image"])
    if cache_path.exists():
        with np.load(cache_path) as cached:
            return {
                "num_components": int(cached["num_components"].item()),
                "mask_coverage": float(cached["mask_coverage"].item()),
            }
    mask = get_pseudo_mask(row)
    with np.load(cache_path) as cached:
        return {
            "num_components": int(cached["num_components"].item()),
            "mask_coverage": float(cached["mask_coverage"].item()),
        }

## Pseudo-mask-guided RGB view

Applies M10's soft pseudo-mask to the P0 global image to produce a lesion-emphasised **RGB**
view (kept at 3 channels, not a 4th channel, so ImageNet-pretrained MaxViT weights remain
directly compatible with the shared backbone — this is different from M09/M10's own 4-channel
approach, which added a separate guidance channel rather than emphasising the RGB image
itself). Emphasis is a simple, deterministic soft blend: pixels under high mask coverage are
brightened toward their original colour with reduced background attenuation, implemented as a
fixed per-pixel linear blend between the original image and itself scaled by the mask, so
still-3-channel RGB comes out the other end.

In [ ]:
GUIDED_VIEW_BACKGROUND_ATTENUATION = 0.35  # Fixed, not tuned against validation results.


def make_guided_view(rgb_224: np.ndarray, pseudo_mask_224: np.ndarray) -> np.ndarray:
    """Blend: full brightness where mask==1, attenuated background where mask==0."""
    mask_3ch = pseudo_mask_224[:, :, None]
    attenuation = GUIDED_VIEW_BACKGROUND_ATTENUATION + (1.0 - GUIDED_VIEW_BACKGROUND_ATTENUATION) * mask_3ch
    guided = (rgb_224.astype(np.float32) * attenuation).clip(0, 255).astype(np.uint8)
    return guided


print(f"Guided-view blend ready (background attenuation={GUIDED_VIEW_BACKGROUND_ATTENUATION}, fixed).")

## 9. Visual sanity checks

Combined crop + mask diagnostics on a fixed random sample covering every grade, before any
training happens.

In [ ]:
sanity_sample_parts = []
per_grade_n = 60
for grade in range(NUM_CLASSES):
    grade_rows = train_df[train_df["level"] == grade]
    sanity_sample_parts.append(grade_rows.sample(n=min(per_grade_n, len(grade_rows)), random_state=SEED))
sanity_df = pd.concat(sanity_sample_parts, ignore_index=True)

records = []
for _, row in tqdm(sanity_df.iterrows(), total=len(sanity_df), desc="visual sanity check"):
    crop_metadata = get_crop_metadata(row)
    mask_metadata = get_mask_metadata(row)
    pseudo_mask = get_pseudo_mask(row)

    assert pseudo_mask.shape == (IMAGE_SIZE, IMAGE_SIZE)
    assert np.isfinite(pseudo_mask).all()
    assert 0.0 <= pseudo_mask.min() and pseudo_mask.max() <= 1.0
    assert 0 <= crop_metadata["crop_x1"] < crop_metadata["crop_x2"] <= DETECTION_SIZE
    assert 0 <= crop_metadata["crop_y1"] < crop_metadata["crop_y2"] <= DETECTION_SIZE

    records.append({
        "grade": int(row["level"]),
        "num_candidate_components": crop_metadata["num_candidate_components"],
        "fallback_used": crop_metadata["fallback_used"],
        "mask_coverage": mask_metadata["mask_coverage"],
    })

records_df = pd.DataFrame(records)

print(f"Sampled images (all grades): {len(records_df)}")
print(f"Mean mask coverage         : {records_df['mask_coverage'].mean():.4f}")
print(f"Mean candidate components  : {records_df['num_candidate_components'].mean():.2f}")
print(f"Crop fallback rate         : {records_df['fallback_used'].mean():.1%}")
print()
print("By grade:")
for grade in range(NUM_CLASSES):
    grade_rows = records_df[records_df["grade"] == grade]
    assert len(grade_rows) > 0, f"No sanity-check examples for grade {grade}."
    print(
        f"  {CLASS_NAMES[grade]:<18s}: mask coverage {grade_rows['mask_coverage'].mean():.4f}, "
        f"components {grade_rows['num_candidate_components'].mean():.2f}, "
        f"fallback {grade_rows['fallback_used'].mean():.1%}"
    )

if records_df["mask_coverage"].mean() > 0.20:
    raise RuntimeError("Pseudo-masks cover too much of the retinal image.")

print()
print("Visual sanity checks: PASSED")

## 10. Three-view paired transforms

All three views must receive **geometrically identical** augmentation — one random decision
applied consistently to the global image, the local crop, and the guided image, otherwise
M12 would be training on three unrelated randomly-perturbed images rather than three
consistent views of the same underlying scene. Colour jitter is sampled once and applied
identically to all three RGB views. Bilinear interpolation throughout (all three are ordinary
RGB images).

In [ ]:
colour_jitter = transforms.ColorJitter(
    brightness=BRIGHTNESS_JITTER, contrast=CONTRAST_JITTER,
    saturation=SATURATION_JITTER, hue=HUE_JITTER,
)


class ThreeViewTransform:
    """Samples one geometric decision and one colour-jitter decision, applies both
    identically to all three views."""

    def __init__(self, train_mode: bool):
        self.train_mode = train_mode

    def __call__(self, global_img: Image.Image, local_img: Image.Image, guided_img: Image.Image):
        if not self.train_mode:
            return tuple(
                TF.normalize(TF.to_tensor(img), mean=list(VIT_MEAN), std=list(VIT_STD))
                for img in (global_img, local_img, guided_img)
            )

        do_flip = torch.rand(1).item() < HORIZONTAL_FLIP_PROBABILITY
        angle, translate, scale, shear = transforms.RandomAffine.get_params(
            degrees=[-ROTATION_DEGREES, ROTATION_DEGREES],
            translate=[TRANSLATE_FRACTION, TRANSLATE_FRACTION],
            scale_ranges=[SCALE_MIN, SCALE_MAX],
            shears=None,
            img_size=[IMAGE_SIZE, IMAGE_SIZE],
        )
        jitter_params = transforms.ColorJitter.get_params(
            colour_jitter.brightness, colour_jitter.contrast,
            colour_jitter.saturation, colour_jitter.hue,
        )
        fn_idx, brightness_factor, contrast_factor, saturation_factor, hue_factor = jitter_params

        views = []
        for img in (global_img, local_img, guided_img):
            if do_flip:
                img = TF.hflip(img)
            img = TF.affine(
                img, angle=angle, translate=translate, scale=scale, shear=shear,
                interpolation=transforms.InterpolationMode.BILINEAR, fill=0,
            )
            for fn_id in fn_idx:
                if fn_id == 0 and brightness_factor is not None:
                    img = TF.adjust_brightness(img, brightness_factor)
                elif fn_id == 1 and contrast_factor is not None:
                    img = TF.adjust_contrast(img, contrast_factor)
                elif fn_id == 2 and saturation_factor is not None:
                    img = TF.adjust_saturation(img, saturation_factor)
                elif fn_id == 3 and hue_factor is not None:
                    img = TF.adjust_hue(img, hue_factor)
            tensor = TF.normalize(TF.to_tensor(img), mean=list(VIT_MEAN), std=list(VIT_STD))
            views.append(tensor)

        return tuple(views)


train_transform = ThreeViewTransform(train_mode=True)
eval_transform = ThreeViewTransform(train_mode=False)
print("Three-view paired transform ready (one geometric + one colour decision per example).")

## 11. Dataset class

Returns global/local/guided tensors, label, and identifying/crop/mask metadata. Validation
mode retains the full metadata needed for the failure-overlap analysis later (Section 25).

In [ ]:
class ThreeViewDRDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, train_mode: bool = False):
        self.df = dataframe.reset_index(drop=True)
        self.train_mode = train_mode
        self.transform = train_transform if train_mode else eval_transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]

        image_full = preprocess_p0_full(row["filepath"])
        global_224 = cv2.resize(image_full, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

        crop_metadata = get_crop_metadata(row)
        detection_image = cv2.resize(image_full, (DETECTION_SIZE, DETECTION_SIZE), interpolation=cv2.INTER_AREA)
        local_512 = detection_image[
            crop_metadata["crop_y1"]:crop_metadata["crop_y2"],
            crop_metadata["crop_x1"]:crop_metadata["crop_x2"],
        ]
        local_224 = cv2.resize(local_512, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

        pseudo_mask = get_pseudo_mask(row)
        mask_metadata = get_mask_metadata(row)
        guided_224 = make_guided_view(global_224, pseudo_mask)

        global_tensor, local_tensor, guided_tensor = self.transform(
            Image.fromarray(global_224), Image.fromarray(local_224), Image.fromarray(guided_224),
        )

        assert global_tensor.shape == (3, IMAGE_SIZE, IMAGE_SIZE)
        assert local_tensor.shape == (3, IMAGE_SIZE, IMAGE_SIZE)
        assert guided_tensor.shape == (3, IMAGE_SIZE, IMAGE_SIZE)
        assert torch.isfinite(global_tensor).all()
        assert torch.isfinite(local_tensor).all()
        assert torch.isfinite(guided_tensor).all()

        label = int(row["level"])
        assert 0 <= label <= 4

        return {
            "global_image": global_tensor,
            "local_image": local_tensor,
            "guided_image": guided_tensor,
            "label": label,
            "image_id": row["image"],
            "patient_id": row["patient_id"],
            "eye": row["eye"],
            "index": index,
            "crop_metadata": crop_metadata,
            "mask_metadata": mask_metadata,
        }


def collate_three_view(batch):
    return {
        "global_image": torch.stack([item["global_image"] for item in batch]),
        "local_image": torch.stack([item["local_image"] for item in batch]),
        "guided_image": torch.stack([item["guided_image"] for item in batch]),
        "label": torch.tensor([item["label"] for item in batch], dtype=torch.long),
        "image_id": [item["image_id"] for item in batch],
        "patient_id": [item["patient_id"] for item in batch],
        "eye": [item["eye"] for item in batch],
        "index": torch.tensor([item["index"] for item in batch], dtype=torch.long),
        "crop_metadata": [item["crop_metadata"] for item in batch],
        "mask_metadata": [item["mask_metadata"] for item in batch],
    }


train_ds = ThreeViewDRDataset(train_df, train_mode=True)
val_ds = ThreeViewDRDataset(val_df, train_mode=False)
print(f"train_ds: {len(train_ds):,} | val_ds: {len(val_ds):,}")

## 12. Data loaders

Ordinary shuffling, no `WeightedRandomSampler` — identical to M07/M11. Uses
`PHYSICAL_BATCH_SIZE` (8); gradient accumulation (Section 17) recovers the effective batch of
32.

In [ ]:
train_loader = DataLoader(
    train_ds, batch_size=PHYSICAL_BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker, collate_fn=collate_three_view,
)
val_loader = DataLoader(
    val_ds, batch_size=PHYSICAL_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker, collate_fn=collate_three_view,
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Physical batch: {PHYSICAL_BATCH_SIZE}  Accumulation: {GRAD_ACCUMULATION_STEPS}  "
      f"Effective batch: {EFFECTIVE_BATCH_SIZE}")

## 13. Class weights

Balanced class weights computed from the training split, clipped to [0.5, 2.5] — unchanged
from M07.

In [ ]:
raw_weights = compute_class_weight(
    class_weight=CLASS_WEIGHT_METHOD, classes=np.arange(NUM_CLASSES), y=train_df["level"].to_numpy(),
)
clipped_weights = np.clip(raw_weights, CLASS_WEIGHT_CLIP_MIN, CLASS_WEIGHT_CLIP_MAX)
class_weights = torch.tensor(clipped_weights, dtype=torch.float32, device=device)
print("Raw balanced weights:", np.round(raw_weights, 3))
print("Clipped weights:", np.round(clipped_weights, 3))

## 14. Combined three-view model

**One shared MaxViT-Tiny backbone** processes all three views — concatenated along the batch
dimension (3×B) in a single forward call, then split back into three chunks and their pooled
embeddings concatenated (`feature_dim * 3`) before a dropout layer and one linear classifier.
No independent per-view backbones (that would triple parameters and confound the comparison
against M07/M09/M10/M11), no additional fusion network beyond concatenation + dropout +
linear, matching the spec exactly.

For checkpoint reloading during validation, `build_model(pretrained=False)` is used since the
trained checkpoint supplies the weights — loading ImageNet weights first would be wasted work
and, worse, would leave stale batch-norm/statistics from a differently-initialised backbone
before `load_state_dict()` overwrites them (harmless here since `load_state_dict` overwrites
everything, but wasteful and slower).

In [ ]:
class CombinedGuidanceMaxViT(nn.Module):
    def __init__(
        self,
        model_name: str = MODEL_NAME,
        num_classes: int = NUM_CLASSES,
        pretrained: bool = True,
        dropout: float = CLASSIFIER_DROPOUT,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, num_classes=0, global_pool="avg",
        )
        feature_dim = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim * 3, num_classes),
        )

    def forward(
        self, global_image: torch.Tensor, local_image: torch.Tensor, guided_image: torch.Tensor,
    ) -> torch.Tensor:
        batch_size = global_image.shape[0]

        combined = torch.cat([global_image, local_image, guided_image], dim=0)
        combined_features = self.backbone(combined)

        if not torch.isfinite(combined_features).all():
            raise RuntimeError("Non-finite features detected.")

        global_features, local_features, guided_features = combined_features.chunk(3, dim=0)
        assert global_features.shape[0] == batch_size
        assert local_features.shape[0] == batch_size
        assert guided_features.shape[0] == batch_size

        fused_features = torch.cat([global_features, local_features, guided_features], dim=1)

        logits = self.classifier(fused_features)
        if not torch.isfinite(logits).all():
            raise RuntimeError("Non-finite classifier logits detected.")

        return logits


def build_model(pretrained: bool = True) -> CombinedGuidanceMaxViT:
    return CombinedGuidanceMaxViT(
        model_name=MODEL_NAME, num_classes=NUM_CLASSES,
        pretrained=pretrained, dropout=CLASSIFIER_DROPOUT,
    ).to(device)


model = build_model(pretrained=True)
total_parameters = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}  (shared backbone, three views, dropout={CLASSIFIER_DROPOUT})")
print(f"Total parameters: {total_parameters:,}")

previous_training_mode = model.training
model.eval()
try:
    with torch.no_grad():
        dummy_global = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
        dummy_local = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
        dummy_guided = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
        output = model(dummy_global, dummy_local, dummy_guided)
finally:
    model.train(previous_training_mode)

assert output.shape == (2, NUM_CLASSES)
print("Three-view dummy forward test: PASSED")

## 15. Loss functions

Exactly M07's loss, unmodified: clipped class-weighted cross-entropy plus a normalised
expected squared grade-distance penalty (weight 0.5).

In [ ]:
class OrdinalPenalisedLoss(nn.Module):
    def __init__(
        self, num_classes: int = NUM_CLASSES, ordinal_weight: float = ORDINAL_PENALTY_WEIGHT,
        class_weights: torch.Tensor = None, label_smoothing: float = 0.0,
    ):
        super().__init__()
        self.ordinal_weight = ordinal_weight
        self.max_squared_distance = float((num_classes - 1) ** 2)
        self.cross_entropy = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
        self.register_buffer("grade_values", torch.arange(num_classes, dtype=torch.float32))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = self.cross_entropy(logits, targets)
        probabilities = F.softmax(logits, dim=1)
        target_grades = targets.to(dtype=probabilities.dtype).unsqueeze(1)
        squared_distances = (self.grade_values.unsqueeze(0) - target_grades).pow(2)
        normalised_distances = squared_distances / self.max_squared_distance
        ordinal_loss = (probabilities * normalised_distances).sum(dim=1).mean()
        return ce_loss + self.ordinal_weight * ordinal_loss


criterion = OrdinalPenalisedLoss(
    num_classes=NUM_CLASSES, ordinal_weight=ORDINAL_PENALTY_WEIGHT,
    class_weights=class_weights, label_smoothing=LABEL_SMOOTHING,
).to(device)
LOSS_FUNCTION_LABEL = "clipped_weighted_ce_plus_normalised_expected_squared_ordinal_penalty"
print("Loss: clipped weighted CE + 0.5 x normalised ordinal penalty (unchanged from M07)")

## 16. Metrics

Metric-computation helpers used by both the per-epoch training loop (Section 17-19) and the
final validation evaluation (Section 21).

In [ ]:
def compute_epoch_metrics(labels: list, preds: list) -> dict:
    labels_arr = np.array(labels)
    preds_arr = np.array(preds)
    absolute_error = np.abs(preds_arr - labels_arr)

    qwk = cohen_kappa_score(labels, preds, weights="quadratic")
    if not np.isfinite(qwk):
        raise RuntimeError("QWK became NaN or infinity during the epoch.")
    mae = float(absolute_error.mean())
    if not np.isfinite(mae):
        raise RuntimeError("MAE became NaN or infinity during the epoch.")

    return {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "qwk": qwk,
        "mae": mae,
        "within_one_grade_accuracy": float((absolute_error <= 1).mean()),
        "large_grade_error_rate": float((absolute_error > 1).mean()),
    }


def checkpoint_improved(
    val_qwk: float, val_macro_f1: float, val_loss: float,
    best_qwk: float, best_macro_f1: float, best_loss: float,
    tolerance: float = QWK_TOLERANCE,
) -> bool:
    """Selection rule: highest QWK, then highest macro-F1, then lower validation loss."""
    if val_qwk > best_qwk + tolerance:
        return True
    if abs(val_qwk - best_qwk) <= tolerance:
        if val_macro_f1 > best_macro_f1 + 1e-6:
            return True
        if abs(val_macro_f1 - best_macro_f1) <= 1e-6 and val_loss < best_loss - 1e-6:
            return True
    return False


print("Metric helpers ready.")

## 17. Corrected AMP training loop

Gradient accumulation with correct scaling of the final, possibly-incomplete accumulation
group; non-finite checks on inputs, logits, loss, and probabilities (features are checked
inside the model itself, Section 14). Gradients are **always** unscaled via
`scaler.unscale_()` at each accumulation boundary, then checked for finiteness — but that
check no longer raises. If gradients are finite, clipping and the optimiser step proceed
normally. If a non-finite gradient is found (an AMP overflow), gradient clipping is skipped
for that step, but `scaler.step(optimizer)` is still called: `GradScaler` itself detects the
overflow internally and silently no-ops the actual parameter update, so calling it is what
makes the skip *safe* rather than corrupting the weights — raising an exception at that point
instead (as an earlier, uncorrected version of this loop did, and as reproduced the exact
failure mode seen in M11) would abort training on what is normal, expected mixed-precision
behaviour, not a genuine training failure. Each skip is counted, printed with the scale
transition, and the running total is returned as `skipped_optimizer_steps` for the training
history.

In [ ]:
def run_epoch(
    model: nn.Module,
    loader,
    optimizer=None,
    train_mode: bool = True,
    grad_clip: float = GRAD_CLIP,
    accumulation_steps: int = GRAD_ACCUMULATION_STEPS,
):
    if train_mode and optimizer is None:
        raise ValueError("An optimizer must be supplied during training.")

    model.train() if train_mode else model.eval()

    total_loss = 0.0
    total_samples = 0
    skipped_optimizer_steps = 0

    all_labels = []
    all_preds = []
    all_probs = []
    all_logits = []
    all_indices = []

    description = "train" if train_mode else "eval"

    if train_mode:
        optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(loader, leave=False, desc=description)):
        global_images = batch["global_image"].to(device, non_blocking=True)
        local_images = batch["local_image"].to(device, non_blocking=True)
        guided_images = batch["guided_image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if not (
            torch.isfinite(global_images).all()
            and torch.isfinite(local_images).all()
            and torch.isfinite(guided_images).all()
        ):
            raise RuntimeError("Non-finite input images detected.")

        with torch.set_grad_enabled(train_mode):
            with torch.autocast(
                device_type=device.type,
                enabled=use_amp,
            ):
                logits = model(
                    global_images,
                    local_images,
                    guided_images,
                )
                loss = criterion(logits, labels)

            if not torch.isfinite(logits).all():
                raise RuntimeError("Non-finite logits detected.")

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss detected.")

        if train_mode:
            group_start = (
                step // accumulation_steps
            ) * accumulation_steps

            steps_in_current_group = min(
                accumulation_steps,
                len(loader) - group_start,
            )

            scaled_loss = loss / steps_in_current_group
            scaler.scale(scaled_loss).backward()

            is_accumulation_boundary = (
                ((step + 1) % accumulation_steps == 0)
                or ((step + 1) == len(loader))
            )

            if is_accumulation_boundary:
                scaler.unscale_(optimizer)

                gradients_finite = True

                for parameter in model.parameters():
                    if (
                        parameter.grad is not None
                        and not torch.isfinite(parameter.grad).all()
                    ):
                        gradients_finite = False
                        break

                scale_before = scaler.get_scale()

                if gradients_finite:
                    if grad_clip > 0:
                        nn.utils.clip_grad_norm_(
                            model.parameters(),
                            grad_clip,
                        )

                    scaler.step(optimizer)
                    scaler.update()

                else:
                    if not use_amp:
                        raise RuntimeError(
                            "Non-finite gradient detected without AMP."
                        )

                    # GradScaler detects the overflow and skips the update.
                    scaler.step(optimizer)
                    scaler.update()

                    skipped_optimizer_steps += 1

                    print(
                        "\nAMP skipped one optimiser step because "
                        "non-finite gradients were detected. "
                        f"Scale: {scale_before:g} -> "
                        f"{scaler.get_scale():g}"
                    )

                optimizer.zero_grad(set_to_none=True)

        probabilities = F.softmax(logits.detach(), dim=1)

        if not torch.isfinite(probabilities).all():
            raise RuntimeError(
                "Non-finite probabilities detected."
            )

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_preds.extend(
            probabilities.argmax(dim=1).cpu().tolist()
        )
        all_labels.extend(
            labels.detach().cpu().tolist()
        )
        all_probs.extend(
            probabilities.cpu().tolist()
        )
        all_logits.extend(
            logits.detach().cpu().tolist()
        )
        all_indices.extend(
            batch["index"].cpu().tolist()
        )

    if total_samples == 0:
        raise RuntimeError(
            "The DataLoader produced no samples."
        )

    epoch_metrics = compute_epoch_metrics(
        all_labels,
        all_preds,
    )

    return {
        "loss": total_loss / total_samples,
        "skipped_optimizer_steps": skipped_optimizer_steps,
        "labels": all_labels,
        "preds": all_preds,
        "probs": all_probs,
        "logits": all_logits,
        "indices": all_indices,
        **epoch_metrics,
    }

## 18. Model-selection rule

The checkpoint rule (`checkpoint_improved()`, defined in Section 16) is:

1. Highest validation QWK wins.
2. If QWK is tied (within `QWK_TOLERANCE`), highest validation macro-F1 wins.
3. If macro-F1 is also tied, lower validation loss wins.

Only validation data is used for model selection — the internal test set is never loaded
anywhere in this notebook.

## 19. Training

`AdamW` with M07's linear warm-up then cosine decay, stepped once per epoch (not per
mini-batch, not per optimiser step under accumulation). `skipped_optimizer_steps` is recorded
in the per-epoch history, as required.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

warmup_scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=max(1, MAX_EPOCHS - WARMUP_EPOCHS))
scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[WARMUP_EPOCHS])

SCHEDULER_CONFIG = {
    "name": "SequentialLR", "warmup_scheduler": "LinearLR",
    "warmup_start_factor": 0.1, "warmup_end_factor": 1.0,
    "warmup_epochs": WARMUP_EPOCHS, "main_scheduler": "CosineAnnealingLR",
}
print(f"Optimizer: AdamW(lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler: {WARMUP_EPOCHS}-epoch linear warm-up -> CosineAnnealingLR (stepped per epoch)")

In [ ]:
history = []
best_qwk = -1.0
best_macro_f1 = -1.0
best_loss = float("inf")
patience_left = EARLY_STOPPING_PATIENCE

print("=" * 70)
print(f"  {RUN_NAME}")
print("=" * 70)
print(f"{'Epoch':<8} {'Tr Loss':>10} {'Val Loss':>10} {'Val QWK':>8} {'Val F1':>8} {'Val MAE':>8} {'Skip':>5}")
print("-" * 70)

for epoch in range(1, MAX_EPOCHS + 1):
    lr_used = optimizer.param_groups[0]["lr"]

    train_metrics = run_epoch(model, train_loader, optimizer=optimizer, train_mode=True)
    val_metrics = run_epoch(model, val_loader, train_mode=False)
    scheduler.step()

    epoch_record = {
        "epoch": epoch,
        "learning_rate": lr_used,
        "train_loss": train_metrics["loss"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_qwk": val_metrics["qwk"],
        "val_mae": val_metrics["mae"],
        "val_within_one_grade_accuracy": val_metrics["within_one_grade_accuracy"],
        "val_large_grade_error_rate": val_metrics["large_grade_error_rate"],
        "skipped_optimizer_steps": train_metrics["skipped_optimizer_steps"],
    }
    history.append(epoch_record)
    pd.DataFrame(history).to_csv(LOG_DIR / "training_history.csv", index=False)

    marker = ""
    if checkpoint_improved(
        val_metrics["qwk"], val_metrics["macro_f1"], val_metrics["loss"],
        best_qwk, best_macro_f1, best_loss,
    ):
        best_qwk = val_metrics["qwk"]
        best_macro_f1 = val_metrics["macro_f1"]
        best_loss = val_metrics["loss"]
        patience_left = EARLY_STOPPING_PATIENCE
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "val_loss": val_metrics["loss"],
                "val_accuracy": val_metrics["accuracy"],
                "val_balanced_accuracy": val_metrics["balanced_accuracy"],
                "val_macro_f1": val_metrics["macro_f1"],
                "val_qwk": val_metrics["qwk"],
                "val_mae": val_metrics["mae"],
            },
            BEST_CHECKPOINT_PATH,
        )
        marker = "  [best]"
    else:
        patience_left -= 1

    print(
        f"{epoch:<8} {train_metrics['loss']:>10.4f} {val_metrics['loss']:>10.4f} "
        f"{val_metrics['qwk']:>8.4f} {val_metrics['macro_f1']:>8.4f} {val_metrics['mae']:>8.4f} "
        f"{train_metrics['skipped_optimizer_steps']:>5d}{marker}"
    )

    if patience_left <= 0:
        print(f"Early stopping at epoch {epoch}.")
        break

print(f"\nBest val QWK = {best_qwk:.4f} (macro-F1 = {best_macro_f1:.4f}, loss = {best_loss:.4f}) "
      f"-> {BEST_CHECKPOINT_PATH.name}")

## 20. Reload selected checkpoint

In [ ]:
best_checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device, weights_only=False)

eval_model = build_model(pretrained=False)
eval_model.load_state_dict(best_checkpoint["model_state"])

print(f"Loaded: {BEST_CHECKPOINT_PATH.name}")
print(f"epoch={best_checkpoint['epoch']}  val_qwk={best_checkpoint['val_qwk']:.4f}  "
      f"val_macro_f1={best_checkpoint['val_macro_f1']:.4f}  val_loss={best_checkpoint['val_loss']:.4f}")

## 21. Final validation evaluation

Re-evaluated fresh on `val_loader` — **not the internal test set**. Extended metrics per the
spec: macro precision/recall, weighted F1, per-class precision/recall/F1, and targeted error
transitions.

In [ ]:
final_val_metrics = run_epoch(eval_model, val_loader, train_mode=False)

val_macro_precision = precision_score(final_val_metrics["labels"], final_val_metrics["preds"], average="macro", zero_division=0)
val_macro_recall = recall_score(final_val_metrics["labels"], final_val_metrics["preds"], average="macro", zero_division=0)
val_weighted_f1 = f1_score(final_val_metrics["labels"], final_val_metrics["preds"], average="weighted", zero_division=0)

per_class_precision = precision_score(final_val_metrics["labels"], final_val_metrics["preds"], average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
per_class_recall = recall_score(final_val_metrics["labels"], final_val_metrics["preds"], average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
per_class_f1 = f1_score(final_val_metrics["labels"], final_val_metrics["preds"], average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
per_class_support = np.bincount(final_val_metrics["labels"], minlength=NUM_CLASSES)

print("=" * 60)
print(f"  {RUN_NAME} -- VALIDATION RESULTS (best checkpoint)")
print("=" * 60)
print(f"  QWK               : {final_val_metrics['qwk']:.4f}")
print(f"  Accuracy          : {final_val_metrics['accuracy']:.4f}")
print(f"  Balanced accuracy : {final_val_metrics['balanced_accuracy']:.4f}")
print(f"  Macro F1          : {final_val_metrics['macro_f1']:.4f}")
print(f"  Weighted F1       : {val_weighted_f1:.4f}")
print(f"  MAE               : {final_val_metrics['mae']:.4f}")
print(f"  Within-1-grade acc: {final_val_metrics['within_one_grade_accuracy']:.4f}")
print(f"  Large-error rate  : {final_val_metrics['large_grade_error_rate']:.4f}")
print()
print(classification_report(
    final_val_metrics["labels"], final_val_metrics["preds"],
    labels=list(range(NUM_CLASSES)), target_names=CLASS_NAMES, digits=3, zero_division=0,
))

In [ ]:
labels_array = np.asarray(final_val_metrics["labels"])
preds_array = np.asarray(final_val_metrics["preds"])
absolute_grade_error = np.abs(preds_array - labels_array)

mild_f1 = float(per_class_f1[1])
moderate_f1 = float(per_class_f1[2])
severe_f1 = float(per_class_f1[3])
pdr_f1 = float(per_class_f1[4])

mild_to_no_dr = int(np.sum((labels_array == 1) & (preds_array == 0)))
severe_to_moderate = int(np.sum((labels_array == 3) & (preds_array == 2)))

print(f"Mild F1              : {mild_f1:.4f}")
print(f"Moderate F1          : {moderate_f1:.4f}")
print(f"Severe F1            : {severe_f1:.4f}")
print(f"PDR F1               : {pdr_f1:.4f}")
print(f"Mild -> No DR errors : {mild_to_no_dr}")
print(f"Severe -> Moderate errors: {severe_to_moderate}")

## 22. Save predictions and logits

`crop_summary.csv` and `mask_summary.csv` record every validation image's crop/mask metadata.
Raw logits/probabilities/labels are saved as `.npy` arrays (in addition to the CSV) for a
possible future M13 calibration notebook.

In [ ]:
crop_summary_rows, mask_summary_rows = [], []
for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="crop/mask summary (validation)"):
    crop_metadata = get_crop_metadata(row)
    mask_metadata = get_mask_metadata(row)
    crop_summary_rows.append({
        "image_id": row["image"], "patient_id": row["patient_id"], "eye": row["eye"],
        "true_grade": int(row["level"]), **crop_metadata,
    })
    mask_summary_rows.append({
        "image_id": row["image"], "patient_id": row["patient_id"], "eye": row["eye"],
        "true_grade": int(row["level"]), **mask_metadata,
    })

crop_summary_df = pd.DataFrame(crop_summary_rows)
mask_summary_df = pd.DataFrame(mask_summary_rows)
crop_summary_df.to_csv(LOG_DIR / "crop_summary.csv", index=False)
mask_summary_df.to_csv(LOG_DIR / "mask_summary.csv", index=False)
print(f"Saved -> {LOG_DIR / 'crop_summary.csv'}")
print(f"Saved -> {LOG_DIR / 'mask_summary.csv'}")

In [ ]:
pred_rows = val_df.iloc[final_val_metrics["indices"]].reset_index(drop=True).copy()
pred_rows["true_label"] = final_val_metrics["labels"]
pred_rows["predicted_label"] = final_val_metrics["preds"]

logits_array = np.array(final_val_metrics["logits"])
probs_array = np.array(final_val_metrics["probs"])
labels_array_ordered = np.array(final_val_metrics["labels"])

np.save(LOG_DIR / "validation_logits.npy", logits_array)
np.save(LOG_DIR / "validation_probabilities.npy", probs_array)
np.save(LOG_DIR / "validation_labels.npy", labels_array_ordered)
pd.DataFrame({"image_id": pred_rows["image"].values}).to_csv(LOG_DIR / "validation_image_ids.csv", index=False)
print(f"Saved -> {LOG_DIR / 'validation_logits.npy'}  {logits_array.shape}")
print(f"Saved -> {LOG_DIR / 'validation_probabilities.npy'}  {probs_array.shape}")
print(f"Saved -> {LOG_DIR / 'validation_labels.npy'}  {labels_array_ordered.shape}")
print(f"Saved -> {LOG_DIR / 'validation_image_ids.csv'}")

for grade in range(NUM_CLASSES):
    pred_rows[f"prob_{grade}"] = probs_array[:, grade]
pred_rows["confidence"] = probs_array.max(axis=1)
pred_rows["correct"] = pred_rows["true_label"] == pred_rows["predicted_label"]
pred_rows["absolute_grade_error"] = (pred_rows["predicted_label"] - pred_rows["true_label"]).abs()
pred_rows = pred_rows.rename(columns={"image": "image_id"})

assert pred_rows["image_id"].is_unique, "Validation prediction image identifiers are not unique."
assert crop_summary_df["image_id"].is_unique, "Crop summary image identifiers are not unique."
assert mask_summary_df["image_id"].is_unique, "Mask summary image identifiers are not unique."

crop_cols = crop_summary_df[["image_id", "crop_x1", "crop_y1", "crop_x2", "crop_y2", "fallback_used"]].rename(
    columns={"fallback_used": "crop_fallback"}
)
mask_cols = mask_summary_df[["image_id", "mask_coverage", "num_components"]].rename(
    columns={"num_components": "number_of_components"}
)
pred_rows = pred_rows.merge(crop_cols, on="image_id", how="left", validate="one_to_one")
pred_rows = pred_rows.merge(mask_cols, on="image_id", how="left", validate="one_to_one")

if pred_rows[["crop_x1", "mask_coverage"]].isna().any().any():
    raise RuntimeError("Some validation predictions are missing crop/mask metadata.")

prediction_columns = [
    "image_id", "true_label", "predicted_label", "confidence",
    "prob_0", "prob_1", "prob_2", "prob_3", "prob_4",
    "absolute_grade_error", "correct",
    "crop_x1", "crop_y1", "crop_x2", "crop_y2", "crop_fallback",
    "mask_coverage", "number_of_components",
]
pred_rows[prediction_columns].to_csv(LOG_DIR / "validation_predictions.csv", index=False)
print(f"Saved -> {LOG_DIR / 'validation_predictions.csv'}  ({len(pred_rows):,} rows)")

In [ ]:
experiment_config = {
    "experiment_id": EXPERIMENT_ID,
    "run_name": RUN_NAME,
    "experiment_type": EXPERIMENT_TYPE,
    "reference_experiment": REFERENCE_EXPERIMENT,
    "template_experiment": TEMPLATE_EXPERIMENT,
    "predecessor_experiment": PREDECESSOR_EXPERIMENT,
    "experimental_change": "combine_local_crop_and_pseudo_mask_guided_view_with_global_view_shared_backbone_fusion",
    "model_name": MODEL_NAME,
    "pretrained": True,
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "architecture": "shared_backbone_three_view_feature_concatenation_with_dropout",
    "classifier_dropout": CLASSIFIER_DROPOUT,
    "physical_batch_size": PHYSICAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUMULATION_STEPS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "optimizer": "AdamW",
    "warmup_epochs": WARMUP_EPOCHS,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "use_wrs": USE_WRS,
    "sampler": "shuffle",
    "loss_function": LOSS_FUNCTION_LABEL,
    "label_smoothing": LABEL_SMOOTHING,
    "class_weight_method": CLASS_WEIGHT_METHOD,
    "class_weight_clip_min": CLASS_WEIGHT_CLIP_MIN,
    "class_weight_clip_max": CLASS_WEIGHT_CLIP_MAX,
    "raw_class_weights": raw_weights.tolist(),
    "clipped_class_weights": clipped_weights.tolist(),
    "ordinal_penalty_type": "normalised_expected_squared_grade_distance",
    "ordinal_penalty_weight": ORDINAL_PENALTY_WEIGHT,
    "grad_clip": GRAD_CLIP,
    "qwk_tolerance": QWK_TOLERANCE,
    "selection_metric": "validation_qwk",
    "selection_tiebreak_1": "validation_macro_f1",
    "selection_tiebreak_2": "validation_loss_lower_is_better",
    "training_mode": TRAINING_MODE,
    "preprocessing": "P0",
    "scheduler": SCHEDULER_CONFIG,
    "normalisation_mean": list(VIT_MEAN),
    "normalisation_std": list(VIT_STD),
    "training_augmentation": {
        "policy": "safe_retinal_augmentation_paired_three_view",
        "horizontal_flip_probability": HORIZONTAL_FLIP_PROBABILITY,
        "rotation_degrees": ROTATION_DEGREES,
        "brightness_jitter": BRIGHTNESS_JITTER,
        "contrast_jitter": CONTRAST_JITTER,
        "saturation_jitter": SATURATION_JITTER,
        "hue_jitter": HUE_JITTER,
        "translation_fraction": TRANSLATE_FRACTION,
        "scale_min": SCALE_MIN,
        "scale_max": SCALE_MAX,
    },
    "guidance_pipeline": {
        "method": "deterministic_classical_image_processing",
        "expert_annotations_used": False,
        "validation_labels_used_to_tune_detector": False,
        "detection_size": DETECTION_SIZE,
        "clahe_clip_limit": CLAHE_CLIP_LIMIT,
        "clahe_tile_grid_size": list(CLAHE_TILE_GRID_SIZE),
        "background_sigma": BACKGROUND_SIGMA,
        "dark_response_percentile": DARK_RESPONSE_PERCENTILE,
        "min_component_area": MIN_COMPONENT_AREA,
        "max_component_area": MAX_COMPONENT_AREA,
        "max_aspect_ratio": MAX_ASPECT_RATIO,
        "max_candidate_components": MAX_CANDIDATE_COMPONENTS,
        "local_crop_size": LOCAL_CROP_SIZE,
        "guided_view_background_attenuation": GUIDED_VIEW_BACKGROUND_ATTENUATION,
        "m11_crop_cache_reused": crop_cache_valid,
        "m10_mask_cache_reused": mask_cache_valid,
    },
    "total_parameters": total_parameters,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "num_workers": NUM_WORKERS,
    "mixed_precision": use_amp,
    "device": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "timm_version": timm.__version__,
    "train_csv": str(TRAIN_CSV),
    "val_csv": str(VAL_CSV),
    "internal_test_loaded": False,
}
with open(LOG_DIR / "config.json", "w") as f:
    json.dump(experiment_config, f, indent=2)
print(f"Saved -> {LOG_DIR / 'config.json'}")

best_metrics = {
    "epoch": best_checkpoint["epoch"],
    "val_loss": best_checkpoint["val_loss"],
    "val_accuracy": best_checkpoint["val_accuracy"],
    "val_balanced_accuracy": best_checkpoint["val_balanced_accuracy"],
    "val_macro_f1": best_checkpoint["val_macro_f1"],
    "val_qwk": best_checkpoint["val_qwk"],
    "val_mae": best_checkpoint["val_mae"],
}
with open(LOG_DIR / "best_metrics.json", "w") as f:
    json.dump(best_metrics, f, indent=2)
print(f"Saved -> {LOG_DIR / 'best_metrics.json'}")

validation_metrics = {
    "qwk": final_val_metrics["qwk"],
    "accuracy": final_val_metrics["accuracy"],
    "balanced_accuracy": final_val_metrics["balanced_accuracy"],
    "macro_f1": final_val_metrics["macro_f1"],
    "weighted_f1": val_weighted_f1,
    "macro_precision": val_macro_precision,
    "macro_recall": val_macro_recall,
    "mean_absolute_grade_error": final_val_metrics["mae"],
    "within_one_grade_accuracy": final_val_metrics["within_one_grade_accuracy"],
    "large_grade_error_rate": final_val_metrics["large_grade_error_rate"],
    "per_class_precision": per_class_precision.tolist(),
    "per_class_recall": per_class_recall.tolist(),
    "per_class_f1": per_class_f1.tolist(),
    "per_class_support": per_class_support.tolist(),
    "mild_f1": mild_f1,
    "moderate_f1": moderate_f1,
    "severe_f1": severe_f1,
    "pdr_f1": pdr_f1,
    "mild_to_no_dr": mild_to_no_dr,
    "severe_to_moderate": severe_to_moderate,
}
with open(LOG_DIR / "validation_metrics.json", "w") as f:
    json.dump(validation_metrics, f, indent=2)
print(f"Saved -> {LOG_DIR / 'validation_metrics.json'}")

## 23. Confusion matrix and training curves

In [ ]:
cm_counts = confusion_matrix(final_val_metrics["labels"], final_val_metrics["preds"])
cm_normalised = cm_counts.astype(float) / cm_counts.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax, data, fmt, title in [
    (axes[0], cm_counts, "d", f"{RUN_NAME} -- counts"),
    (axes[1], cm_normalised, ".2f", f"{RUN_NAME} -- normalised"),
]:
    im = ax.imshow(data, interpolation="nearest", cmap="Purples")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title, fontweight="bold")
    thresh = data.max() / 2.0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, format(data[i, j], fmt), ha="center", va="center",
                     color="white" if data[i, j] > thresh else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "confusion_matrix.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'confusion_matrix.png'}")

In [ ]:
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train", color="#3498DB")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validation", color="#E74C3C")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title(f"{RUN_NAME} -- Loss", fontweight="bold")
axes[0].legend(); axes[0].spines[["top", "right"]].set_visible(False)

axes[1].plot(history_df["epoch"], history_df["val_qwk"], label="Validation QWK", color="#E74C3C")
axes[1].plot(history_df["epoch"], history_df["val_mae"], label="Validation MAE", color="#8E44AD")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Metric value")
axes[1].set_title(f"{RUN_NAME} -- QWK / MAE", fontweight="bold")
axes[1].legend(); axes[1].spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "training_curves.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'training_curves.png'}")

if history_df["skipped_optimizer_steps"].sum() > 0:
    print(f"\nTotal skipped optimiser steps across training: {int(history_df['skipped_optimizer_steps'].sum())}")
    print("(AMP-detected gradient overflows, safely skipped - not a training failure.)")

## 24. Comparison with M07 (reference), M09, M10, and M11

**M07 is the training reference; M09/M10/M11 are the individual-component comparisons.**
Loads each notebook's **saved JSON results**, never their checkpoints. Accuracy alone is
never used to conclude M12 is "better" — QWK is primary, with the full metric set reported
alongside it.

In [ ]:
def _load_other_metrics(log_dir: Path, other_name: str):
    best_path = log_dir / "best_metrics.json"
    val_path = log_dir / "validation_metrics.json"
    if not (best_path.exists() and val_path.exists()):
        print(f"{other_name} results not found at {log_dir}; skipping from the comparison table.")
        return None
    with open(best_path) as f:
        best = json.load(f)
    with open(val_path) as f:
        val = json.load(f)
    return {
        "qwk": best["val_qwk"],
        "macro_f1": best["val_macro_f1"],
        "balanced_accuracy": best["val_balanced_accuracy"],
        "accuracy": best["val_accuracy"],
        "mae": val.get("mean_absolute_grade_error", val.get("mae", float("nan"))),
        "large_grade_error_rate": val.get("large_grade_error_rate", float("nan")),
        "mild_f1": val.get("mild_f1", val["per_class_f1"][1]),
        "moderate_f1": val.get("moderate_f1", val["per_class_f1"][2]),
        "severe_f1": val.get("severe_f1", val["per_class_f1"][3]),
        "pdr_f1": val.get("pdr_f1", val["per_class_f1"][4]),
        "mild_to_no_dr": val.get("mild_to_no_dr", float("nan")),
        "severe_to_moderate": val.get("severe_to_moderate", float("nan")),
    }


m12_metrics_row = {
    "qwk": final_val_metrics["qwk"], "macro_f1": final_val_metrics["macro_f1"],
    "balanced_accuracy": final_val_metrics["balanced_accuracy"], "accuracy": final_val_metrics["accuracy"],
    "mae": final_val_metrics["mae"], "large_grade_error_rate": final_val_metrics["large_grade_error_rate"],
    "mild_f1": mild_f1, "moderate_f1": moderate_f1, "severe_f1": severe_f1, "pdr_f1": pdr_f1,
    "mild_to_no_dr": mild_to_no_dr, "severe_to_moderate": severe_to_moderate,
}

metric_labels = [
    ("qwk", "QWK"), ("macro_f1", "Macro-F1"), ("balanced_accuracy", "Balanced accuracy"),
    ("accuracy", "Accuracy"), ("mae", "MAE"), ("large_grade_error_rate", "Large-grade error rate"),
    ("mild_f1", "Mild F1"), ("moderate_f1", "Moderate F1"), ("severe_f1", "Severe F1"),
    ("pdr_f1", "PDR F1"), ("mild_to_no_dr", "Mild -> No DR count"), ("severe_to_moderate", "Severe -> Moderate count"),
]

comparison_table = {"Metric": [label for _, label in metric_labels]}
other_experiments = [("M07", M07_LOG_DIR), ("M09", M09_LOG_DIR), ("M10", M10_LOG_DIR), ("M11", M11_LOG_DIR)]
other_metrics_by_name = {}
for other_name, other_dir in other_experiments:
    other_metrics = _load_other_metrics(other_dir, other_name)
    other_metrics_by_name[other_name] = other_metrics
    comparison_table[other_name] = (
        [other_metrics[key] for key, _ in metric_labels] if other_metrics is not None
        else [np.nan] * len(metric_labels)
    )
comparison_table["M12"] = [m12_metrics_row[key] for key, _ in metric_labels]

comparison_df = pd.DataFrame(comparison_table)
print(comparison_df.to_string(index=False))
comparison_df.to_csv(LOG_DIR / "m07_m12_comparison.csv", index=False)
print(f"Saved -> {LOG_DIR / 'm07_m12_comparison.csv'}")

for other_name, _ in [("M09", None), ("M10", None), ("M11", None)]:
    if other_metrics_by_name[other_name] is not None:
        sub_df = comparison_df[["Metric", other_name, "M12"]].copy()
        sub_df[f"{other_name}_to_M12_difference"] = sub_df["M12"] - sub_df[other_name]
        sub_df.to_csv(LOG_DIR / f"{other_name.lower()}_m12_comparison.csv", index=False)
        print(f"Saved -> {LOG_DIR / f'{other_name.lower()}_m12_comparison.csv'}")

fig, ax = plt.subplots(figsize=(11, 0.5 * len(comparison_df) + 1))
ax.axis("off")
table = ax.table(
    cellText=comparison_df.round(4).astype(str).values, colLabels=comparison_df.columns,
    cellLoc="center", loc="center",
)
table.auto_set_font_size(False); table.set_fontsize(8); table.scale(1, 1.4)
ax.set_title("M07 (reference) / M09 / M10 / M11 / M12 -- combined comparison", fontweight="bold", pad=20)
plt.savefig(FIGURE_DIR / "m07_m12_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'm07_m12_comparison.png'}")

In [ ]:
class_f1_data = {
    name: [m["mild_f1"], m["moderate_f1"], m["severe_f1"], m["pdr_f1"]] if m is not None else [np.nan] * 4
    for name, m in list(other_metrics_by_name.items()) + [("M12", m12_metrics_row)]
}
class_f1_df = pd.DataFrame(class_f1_data, index=["Mild", "Moderate", "Severe", "PDR"])

fig, ax = plt.subplots(figsize=(9, 5))
class_f1_df.plot(kind="bar", ax=ax)
ax.set_ylabel("F1 score")
ax.set_title("Per-class F1 across M07/M09/M10/M11/M12", fontweight="bold")
ax.legend(title="Experiment")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "class_f1_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'class_f1_comparison.png'}")

In [ ]:
component_metrics = ["qwk", "macro_f1", "mae", "large_grade_error_rate"]
component_labels = ["QWK", "Macro-F1", "MAE", "Large-grade error rate"]
component_data = {
    name: [m[key] for key in component_metrics] if m is not None else [np.nan] * len(component_metrics)
    for name, m in list(other_metrics_by_name.items()) + [("M12", m12_metrics_row)]
}
component_df = pd.DataFrame(component_data, index=component_labels)

fig, axes = plt.subplots(1, len(component_metrics), figsize=(16, 4))
for ax, metric_label in zip(axes, component_labels):
    component_df.loc[metric_label].plot(kind="bar", ax=ax, color="#3498DB")
    ax.set_title(metric_label, fontweight="bold")
    ax.set_xlabel("")
plt.suptitle("M07 (reference) / M09 / M10 / M11 / M12 -- component comparison", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "component_comparison.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'component_comparison.png'}")

## 25. Failure-overlap analysis

Loads validation predictions from M09, M10, M11, and M12, matches rows by image identifier,
and analyses where errors overlap or diverge. Column names differ slightly between notebooks
(`image` in M09/M10/M11's prediction CSVs vs `image_id` here), so each is normalised on load.
Skips gracefully if any of M09/M10/M11's predictions aren't available yet.

In [ ]:
def _load_other_predictions(log_dir: Path, other_name: str):
    pred_path = log_dir / "validation_predictions.csv"
    if not pred_path.exists():
        print(f"{other_name} validation_predictions.csv not found; failure-overlap analysis will be incomplete.")
        return None
    df = pd.read_csv(pred_path)
    if "image" in df.columns and "image_id" not in df.columns:
        df = df.rename(columns={"image": "image_id"})
    return df[["image_id", "true_label", "predicted_label"]].rename(
        columns={"predicted_label": f"{other_name.lower()}_predicted_label"}
    )


other_pred_frames = {}
for other_name, other_dir in [("M09", M09_LOG_DIR), ("M10", M10_LOG_DIR), ("M11", M11_LOG_DIR)]:
    other_pred_frames[other_name] = _load_other_predictions(other_dir, other_name)

m12_pred_df = pred_rows[["image_id", "true_label", "predicted_label"]].rename(
    columns={"predicted_label": "m12_predicted_label"}
)

merged = m12_pred_df.copy()
available_others = []
for other_name in ["M09", "M10", "M11"]:
    frame = other_pred_frames[other_name]
    if frame is not None:
        merged = merged.merge(
            frame.drop(columns=["true_label"]), on="image_id", how="inner",
        )
        available_others.append(other_name)

print(f"Failure-overlap analysis available for: {available_others} (+ M12)")
print(f"Matched rows: {len(merged):,} / {len(m12_pred_df):,} M12 validation images")

merged["m12_wrong"] = merged["m12_predicted_label"] != merged["true_label"]
for other_name in available_others:
    col = f"{other_name.lower()}_predicted_label"
    merged[f"{other_name.lower()}_wrong"] = merged[col] != merged["true_label"]

wrong_cols = [f"{name.lower()}_wrong" for name in available_others] + ["m12_wrong"]

In [ ]:
if len(available_others) == 3:
    all_four_wrong = int((merged[wrong_cols].all(axis=1)).sum())
    only_m12_wrong = int((merged["m12_wrong"] & ~merged["m09_wrong"] & ~merged["m10_wrong"] & ~merged["m11_wrong"]).sum())
    m09_m10_wrong_m12_correct = int(
        (merged["m09_wrong"] & merged["m10_wrong"] & ~merged["m12_wrong"]).sum()
    )
    m11_wrong_m12_correct = int((merged["m11_wrong"] & ~merged["m12_wrong"]).sum())
    all_component_correct = int((~merged[wrong_cols].any(axis=1)).sum())
    correct_individually_but_m12_wrong = int(
        (~merged["m09_wrong"] & ~merged["m10_wrong"] & ~merged["m11_wrong"] & merged["m12_wrong"]).sum()
    )

    print(f"All M09-M12 wrong                      : {all_four_wrong}")
    print(f"Only M12 wrong                         : {only_m12_wrong}")
    print(f"M09 & M10 wrong but M12 correct         : {m09_m10_wrong_m12_correct}")
    print(f"M11 wrong but M12 correct               : {m11_wrong_m12_correct}")
    print(f"All component models correct            : {all_component_correct}")
    print(f"Correct individually, made wrong by M12 : {correct_individually_but_m12_wrong}")

    mild_actual_mask = merged["true_label"] == 1
    severe_actual_mask = merged["true_label"] == 3
    common_mild_to_no_dr = int(
        (mild_actual_mask & merged[[f"{n.lower()}_predicted_label" for n in available_others] + ["m12_predicted_label"]].eq(0).all(axis=1)).sum()
    )
    common_severe_to_moderate = int(
        (severe_actual_mask & merged[[f"{n.lower()}_predicted_label" for n in available_others] + ["m12_predicted_label"]].eq(2).all(axis=1)).sum()
    )
    common_large_grade_errors = int(
        (merged[[f"{n.lower()}_predicted_label" for n in available_others] + ["m12_predicted_label"]]
         .sub(merged["true_label"], axis=0).abs().gt(1).all(axis=1)).sum()
    )
    print(f"Common Mild -> No DR errors (all models): {common_mild_to_no_dr}")
    print(f"Common Severe -> Moderate errors (all)   : {common_severe_to_moderate}")
    print(f"Common large-grade errors (all models)   : {common_large_grade_errors}")

    failure_overlap_summary = pd.DataFrame([{
        "all_four_wrong": all_four_wrong,
        "only_m12_wrong": only_m12_wrong,
        "m09_m10_wrong_m12_correct": m09_m10_wrong_m12_correct,
        "m11_wrong_m12_correct": m11_wrong_m12_correct,
        "all_component_models_correct": all_component_correct,
        "correct_individually_but_m12_wrong": correct_individually_but_m12_wrong,
        "common_mild_to_no_dr": common_mild_to_no_dr,
        "common_severe_to_moderate": common_severe_to_moderate,
        "common_large_grade_errors": common_large_grade_errors,
        "matched_rows": len(merged),
    }])
else:
    print("Not all of M09/M10/M11 have predictions available yet - only a partial "
          "failure-overlap summary can be produced.")
    failure_overlap_summary = pd.DataFrame([{
        "matched_rows": len(merged),
        "available_experiments": ",".join(available_others + ["M12"]),
        "note": "Full four-way overlap requires M09, M10, and M11 predictions to all be available.",
    }])

failure_overlap_summary.to_csv(LOG_DIR / "failure_overlap_summary.csv", index=False)
print(f"\nSaved -> {LOG_DIR / 'failure_overlap_summary.csv'}")

In [ ]:
def jaccard(set_a: set, set_b: set) -> float:
    union = set_a | set_b
    if len(union) == 0:
        return float("nan")
    return len(set_a & set_b) / len(union)


error_sets = {"M12": set(merged.loc[merged["m12_wrong"], "image_id"])}
for other_name in available_others:
    error_sets[other_name] = set(merged.loc[merged[f"{other_name.lower()}_wrong"], "image_id"])

jaccard_rows = []
for name_a, name_b in combinations(sorted(error_sets.keys()), 2):
    set_a, set_b = error_sets[name_a], error_sets[name_b]
    jaccard_rows.append({
        "Pair": f"{name_a}-{name_b}",
        "Shared errors": len(set_a & set_b),
        "Union errors": len(set_a | set_b),
        "Jaccard": jaccard(set_a, set_b),
    })
jaccard_df = pd.DataFrame(jaccard_rows)
print(jaccard_df.to_string(index=False))
jaccard_df.to_csv(LOG_DIR / "error_set_jaccard_similarity.csv", index=False)
print(f"Saved -> {LOG_DIR / 'error_set_jaccard_similarity.csv'}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(jaccard_df["Pair"], jaccard_df["Jaccard"], color="#8E44AD")
ax.set_ylabel("Jaccard similarity of error sets")
ax.set_title("Error-set overlap across M09/M10/M11/M12", fontweight="bold")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "error_overlap_upset_or_bar.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'error_overlap_upset_or_bar.png'}")

## 26. Qualitative combined-view analysis

At least two examples per grade, each showing: the original global image, the
detection-selected crop, the pseudo-mask, the pseudo-mask-guided image, the true grade, the
predicted grade, and the model's confidence.

In [ ]:
examples_per_grade = 2
example_rows = []
for grade in range(NUM_CLASSES):
    grade_val_rows = pred_rows[pred_rows["true_label"] == grade]
    if len(grade_val_rows) == 0:
        continue
    example_rows.append(grade_val_rows.sample(n=min(examples_per_grade, len(grade_val_rows)), random_state=SEED))
example_pred_df = pd.concat(example_rows, ignore_index=True)

# Recover source-row info (filepath) for each sampled image_id.
example_full_df = example_pred_df.merge(
    val_df[["image", "filepath"]].rename(columns={"image": "image_id"}), on="image_id", how="left",
)

fig, axes = plt.subplots(len(example_full_df), 4, figsize=(12, 3 * len(example_full_df)))

for row_idx, row in example_full_df.iterrows():
    image_full = preprocess_p0_full(row["filepath"])
    global_224 = cv2.resize(image_full, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
    pseudo_mask = generate_pseudo_segmentation_mask(image_full)["mask"]
    guided_224 = make_guided_view(global_224, pseudo_mask)

    axes[row_idx, 0].imshow(global_224)
    axes[row_idx, 0].set_title("global image", fontsize=9)

    crop_meta = {
        "crop_x1": int(row["crop_x1"]), "crop_y1": int(row["crop_y1"]),
        "crop_x2": int(row["crop_x2"]), "crop_y2": int(row["crop_y2"]),
    }
    detection_image = cv2.resize(image_full, (DETECTION_SIZE, DETECTION_SIZE), interpolation=cv2.INTER_AREA)
    local_224 = cv2.resize(
        detection_image[crop_meta["crop_y1"]:crop_meta["crop_y2"], crop_meta["crop_x1"]:crop_meta["crop_x2"]],
        (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA,
    )
    axes[row_idx, 1].imshow(local_224)
    axes[row_idx, 1].set_title("detection-selected crop", fontsize=9)

    axes[row_idx, 2].imshow(pseudo_mask, cmap="hot", vmin=0, vmax=1)
    axes[row_idx, 2].set_title(f"pseudo-mask (coverage={pseudo_mask.mean():.3f})", fontsize=9)

    axes[row_idx, 3].imshow(guided_224)
    true_grade_name = CLASS_NAMES[int(row["true_label"])]
    pred_grade_name = CLASS_NAMES[int(row["predicted_label"])]
    axes[row_idx, 3].set_title(
        f"guided view\ntrue={true_grade_name}  pred={pred_grade_name}\nconf={row['confidence']:.2f}",
        fontsize=8,
    )

    for col in range(4):
        axes[row_idx, col].axis("off")

fig.suptitle(
    "Candidate regions and pseudo-masks are deterministic, heuristic image-processing "
    "outputs — not expert or ground-truth lesion annotations.",
    fontsize=10,
)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "combined_view_examples.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved -> {FIGURE_DIR / 'combined_view_examples.png'}")

## 27. Limitations

**The candidate detector and pseudo-segmentation masks are heuristic and are not validated
against expert lesion annotations. Therefore, the generated regions must not be described as
confirmed diabetic-retinopathy lesions.** Throughout this notebook and its outputs, these are
referred to as candidate regions, pseudo-masks, heuristic fine-grained guidance, or
lesion-like structures — never as verified lesions, ground-truth lesion segmentation, or
clinically validated lesion crops.

Further limitations specific to the combined design:

- The three-view fusion triples the per-image feature-extraction cost relative to M07's
  single view (partially offset by the shared backbone, which avoids tripling the parameter
  count, but not the compute).
- The candidate-generation and mask-generation thresholds are fixed heuristics inherited from
  M09/M10/M11, not learned or tuned for this specific three-view combination.
- The guided view's background-attenuation blend (Section: "Pseudo-mask-guided RGB view") is
  itself a fixed, untuned design choice — a different blend strategy might behave
  differently.
- Cache reuse from M09/M10/M11 assumes those notebooks' candidate-generation parameters
  genuinely match M12's (validated in Sections 7-8), but the *underlying source images* are
  assumed unchanged since those caches were built — if the dataset were modified without
  invalidating the cache, this would not be caught automatically here.
- The failure-overlap analysis (Section 25) depends entirely on M09/M10/M11 having already
  been run and their `validation_predictions.csv` being present; if any are missing, that
  analysis is necessarily partial (explicitly reported, not silently ignored).

## 28. Experiment conclusion

To be completed after execution:

- Did combining the components improve QWK over M11?
- Did M12 recover M11's Severe-class loss?
- Did Mild and Moderate F1 improve?
- Did the components reinforce the same errors?
- Did M12 correct complementary errors?
- Did mask coverage or crop properties relate to correctness?
- Did M12 reduce large-grade errors?
- Was the computational cost justified?
- Did training overfit?
- Is M12 preferable to M07 or M11 under the QWK-first rule?